# Bài 4.4 · LoRA so với fine-tune toàn bộ

**SST-2** gồm các câu nhận xét phim tiếng Anh, nhãn tích cực hoặc tiêu cực. Notebook dùng RoBERTa-base (khoảng 125 triệu tham số) và so ba cách học trên cùng 8000 câu train, chấm trên 872 câu validation:

1. **Chỉ học head**: đóng băng toàn bộ encoder.
2. **LoRA**: đóng băng encoder, thêm ma trận hạng thấp $BA$ vào query và value của mọi lớp attention.
3. **Fine-tune toàn bộ**: mọi tham số đều học.

Với mỗi cách ghi số tham số được học, accuracy, thời gian và bộ nhớ GPU đỉnh. Cuối notebook merge LoRA vào trọng số gốc để kiểm logits không đổi, và so kích thước tệp lưu.

Bật **GPU** và **Internet**. Chạy hết khoảng 10 phút.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec('peft') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'peft'], check=True)

import os, time, copy
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('thiết bị:', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')
MODEL, N_TRAIN, EPOCHS, BATCH, MAXLEN = 'roberta-base', 8000, 2, 32, 64

## 1. Dữ liệu

Tokenize một lần, cắt và đệm về 64 token. Câu SST-2 ngắn, nên 64 token giữ gần như trọn câu.

In [ ]:
ds = load_dataset('stanfordnlp/sst2')
train = ds['train'].shuffle(seed=0).select(range(N_TRAIN))
val = ds['validation']
tok = AutoTokenizer.from_pretrained(MODEL)

def encode(split):
    e = tok(list(split['sentence']), truncation=True, max_length=MAXLEN, padding='max_length', return_tensors='pt')
    return list(zip(e['input_ids'], e['attention_mask'], torch.tensor(list(split['label']))))

train_data, val_data = encode(train), encode(val)
print(len(train_data), 'câu train |', len(val_data), 'câu validation | tỉ lệ nhãn 1 ở train:', np.mean(list(train['label'])).round(3))
print(train[0]['sentence'], '→', train[0]['label'])

## 2. Một hàm huấn luyện dùng chung

Ba cách chỉ khác nhau ở **tham số nào được học**; vòng lặp, learning rate schedule và số epoch giữ nguyên, trừ learning rate: fine-tune toàn bộ dùng learning rate nhỏ hơn để không phá trọng số đã học.

In [ ]:
def collate(batch):
    ids, mask, y = zip(*batch)
    return torch.stack(ids), torch.stack(mask), torch.stack(y)

train_dl = DataLoader(train_data, BATCH, shuffle=True, collate_fn=collate)
val_dl = DataLoader(val_data, 128, collate_fn=collate)

def evaluate(model):
    model.eval(); hit = 0
    with torch.inference_mode(), torch.autocast(device, dtype=torch.float16, enabled=device == 'cuda'):
        for ids, mask, y in val_dl:
            hit += (model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits.argmax(1).cpu() == y).sum().item()
    return hit / len(val_data)

def fit(model, lr):
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=EPOCHS * len(train_dl), pct_start=0.1)
    scaler = torch.amp.GradScaler('cuda', enabled=device == 'cuda')
    if device == 'cuda':
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    t, accs = time.time(), []
    for ep in range(EPOCHS):
        model.train()
        for ids, mask, y in train_dl:
            with torch.autocast(device, dtype=torch.float16, enabled=device == 'cuda'):
                loss = model(input_ids=ids.to(device), attention_mask=mask.to(device), labels=y.to(device)).loss
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        accs.append(evaluate(model))
        print(f'  epoch {ep + 1}: val acc {accs[-1]:.3f}')
    mem = torch.cuda.max_memory_allocated() / 2**30 if device == 'cuda' else float('nan')
    return accs[-1], time.time() - t, mem

def trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

new_model = lambda: AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2).to(device)
results = []

## 3. Chỉ học head

Head phân loại của RoBERTa gồm một lớp dense và một lớp ra 2 logit; mọi tham số khác bị đóng băng.

In [ ]:
m = new_model()
for n, p in m.named_parameters():
    p.requires_grad = n.startswith('classifier.')
print(f'tham số được học: {trainable(m):,}')
acc, sec, mem = fit(m, 1e-3)
results.append(('chỉ head', trainable(m), acc, sec, mem))
del m

## 4. LoRA

`r = 8`: mỗi ma trận $W$ của query và value có thêm $B$ ($768 \times 8$) và $A$ ($8 \times 768$). $B$ khởi tạo bằng 0, nên lúc đầu model y hệt bản gốc. Head phân loại vẫn được học đầy đủ (`modules_to_save`).

In [ ]:
cfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.1, target_modules=['query', 'value'],
                 modules_to_save=['classifier'], task_type='SEQ_CLS')
lora = get_peft_model(new_model(), cfg)
lora.print_trainable_parameters()
lora_only = sum(p.numel() for n, p in lora.named_parameters() if 'lora_' in n)
print(f'riêng ma trận LoRA: {lora_only:,} = 12 lớp × 2 ma trận × (768·8 + 8·768)')
acc, sec, mem = fit(lora, 1e-3)
results.append(('LoRA r=8', trainable(lora), acc, sec, mem))

## 5. Fine-tune toàn bộ

In [ ]:
m = new_model()
print(f'tham số được học: {trainable(m):,}')
acc, sec, mem = fit(m, 2e-5)
results.append(('fine-tune toàn bộ', trainable(m), acc, sec, mem))
full_bytes = sum(p.numel() * p.element_size() for p in m.parameters())
del m
if device == 'cuda': torch.cuda.empty_cache()

In [ ]:
res = pd.DataFrame(results, columns=['cách', 'tham số được học', 'val acc', 'giây', 'GPU đỉnh (GB)'])
res['tham số được học'] = res['tham số được học'].map('{:,}'.format)
res.round(3)

## 6. Merge LoRA và kiểm tính tương đương

Sau khi học xong, cộng thẳng $\frac{\alpha}{r} BA$ vào $W$. Model merge không còn nhánh LoRA, chạy nhanh như model gốc, và phải cho **cùng logits** với model chưa merge (chỉ lệch ở mức làm tròn số thực).

In [ ]:
lora.eval()
ids, mask, _ = next(iter(val_dl))
ids, mask = ids[:16].to(device), mask[:16].to(device)
with torch.inference_mode():
    before = lora(input_ids=ids, attention_mask=mask).logits.float()
merged = copy.deepcopy(lora).merge_and_unload().eval()
with torch.inference_mode():
    after = merged(input_ids=ids, attention_mask=mask).logits.float()
print('chênh logits lớn nhất sau merge:', (before - after).abs().max().item())
print('model merge còn lớp LoRA không:', any('lora_' in n for n, _ in merged.named_parameters()))

## 7. Kích thước tệp lưu

Adapter LoRA chỉ chứa $A$, $B$ và head; model fine-tune toàn bộ phải lưu cả 125 triệu tham số.

In [ ]:
lora.save_pretrained('/tmp/lora_adapter')
adapter_mb = sum(os.path.getsize(os.path.join('/tmp/lora_adapter', f)) for f in os.listdir('/tmp/lora_adapter')) / 2**20
print(f'adapter LoRA: {adapter_mb:.1f} MB | model fine-tune toàn bộ (float32): {full_bytes / 2**20:.0f} MB')